# 11 · Check the PFB export fixes

This notebook exports a small cohort as PFB with `exportAsPFB()` and checks two changes to the file HPDS writes. Each change is a pull request in `hms-dbmi/pic-sure`.

| Ticket | Pull request | What the check covers |
|---|---|---|
| ALS-13635 | #494 | The data dictionary rows carry `display`, `dataset` and `description` from the dictionary service. They used to be blank. |
| ALS-13611 | #496 | The `drs_uri` array of a data dictionary row is filled when the dictionary has `drs_uri` metadata for that concept. |

### What you need

- A PIC-SURE stack whose HPDS has both changes. Against a stack without them the export still works, and the checks below report what is missing.
- A token for an account that may export the concepts you pick.
- For the ALS-13611 check, dictionary rows that have `drs_uri` metadata. Without them the check reports "not applicable".

### What a PFB from HPDS contains

The file is Avro. Every record is an `Entity` with four fields: `id`, `name`, `object` and `relations`. The `name` field says which table the record belongs to.

- `Metadata`: the first record. Its `nodes` list names each table and the fields of that table.
- `pic_sure_data_dictionary_<query id>`: one record per exported concept, with `concept_path`, `display`, `dataset`, `description` and `drs_uri`.
- `pic_sure_patients_<query id>`: one record per patient. `patient_id` is a single string. Every other field is an array of strings, one entry per observation.

The query id in a table name has `_` in place of each `-`.

The `concept_path` column of the data dictionary table holds the concept path as the dictionary lists it, backslashes included. The checks match rows on that column.

Avro field names allow only letters, digits and underscores, so a patient field is not named by the concept path itself. HPDS replaces every other character of the path with `_` and prefixes a name that starts with a digit with `_`. The path `\phs123\stroke\` becomes the field `_phs123_stroke_`. When two paths give the same name, the second gets the suffix `_2` and the third `_3`. The `id` of a data dictionary record is the patient field name of its concept.

## Setup

`fastavro` reads the file. It is already a dependency of the `picsure` package, so there is nothing extra to install.

In [ ]:
import os
from dataclasses import dataclass
from getpass import getpass
from pathlib import Path

import pandas as pd
from fastavro import reader

import picsure

### Settings

- `PFB_PATH` is where the export is written. `*.avro` files are ignored by git in this repository.
- `REUSE_EXISTING_FILE` skips the connection and the export and checks the file already at `PFB_PATH`. Set the `PICSURE_PFB_REUSE` environment variable to `1` to turn it on.
- `CONCEPT_PATHS` lists the concepts to export. Leave it empty to take the first `MAX_CONCEPTS` results of a dictionary search for `SEARCH_TERM`.

To see both checks pass, pick at least one concept whose dictionary entry has `drs_uri` metadata. Keep the list short. The export holds one record per patient who has a value for the first concept.

In [ ]:
PFB_PATH = Path(os.environ.get("PICSURE_PFB_PATH", "pfb_export_fixes.avro"))
REUSE_EXISTING_FILE = os.environ.get("PICSURE_PFB_REUSE", "") == "1"

CONCEPT_PATHS = []
SEARCH_TERM = os.environ.get("PICSURE_PFB_SEARCH_TERM", "age")
MAX_CONCEPTS = 5

### Connect

The URL comes from the `PICSURE_URL` environment variable and the token from `PICSURE_TOKEN`. If either is unset the cell asks for it, and the token prompt does not echo. Nothing is read from a file and nothing is stored in the notebook.

Pass the base URL of the stack, for example `https://localhost` for a stack on your own machine. For a stack with a self-signed certificate, set `PICSURE_SSL_VERIFY=false` before starting Jupyter. Leave it unset against any shared environment.

In [ ]:
def read_setting(name, prompt, secret=False):
    """Return an environment variable, or ask for the value when it is unset.

    Args:
        name: The environment variable to read.
        prompt: The text shown when the variable is unset.
        secret: Whether to hide what is typed.

    Returns:
        The value with surrounding whitespace removed.
    """
    value = os.environ.get(name, "").strip()
    if value:
        return value
    return (getpass(prompt) if secret else input(prompt)).strip()


session = None
if not REUSE_EXISTING_FILE:
    session = picsure.connect(
        read_setting("PICSURE_URL", "PIC-SURE URL: "),
        read_setting("PICSURE_TOKEN", "PIC-SURE token: ", secret=True),
    )

## Export the PFB

### Pick the concepts

When `CONCEPT_PATHS` is empty, this cell fills it from a dictionary search.

In [ ]:
if session is not None and not CONCEPT_PATHS:
    matches = session.searchDictionary(
        SEARCH_TERM,
        include_values=False,
        page=0,
        page_size=MAX_CONCEPTS,
    )
    CONCEPT_PATHS = matches["conceptPath"].head(MAX_CONCEPTS).tolist()

CONCEPT_PATHS

### Run the export

The cohort is every patient with a value for the first concept, and the export selects all the listed concepts as columns.

`exportAsPFB()` submits the query to `/picsure/hpds/auth/query` with `expectedResultType` set to `DATAFRAME_PFB`, polls the job, and streams the Avro bytes to disk.

With ALS-13635 the export needs the dictionary service. If HPDS cannot reach it, the job ends in `ERROR` and this cell raises `PicSureQueryError`. Before the change the export finished with blank dictionary columns.

In [ ]:
if session is not None:
    if not CONCEPT_PATHS:
        raise ValueError(
            "No concepts to export. Fill CONCEPT_PATHS or change SEARCH_TERM."
        )
    cohort = picsure.buildClause(
        CONCEPT_PATHS[0],
        picsure.PhenotypicFilterType.REQUIRE,
    )
    query = picsure.buildQuery(
        phenotypicFilter=cohort,
        includeConcepts=CONCEPT_PATHS,
    )
    session.exportAsPFB(query=query, path=PFB_PATH)

print(f"{PFB_PATH}: {PFB_PATH.stat().st_size} bytes")

## Read the PFB

`read_pfb()` tells the tables apart by the `name` of each record. It keeps the data dictionary rows and counts the patient rows. A file with no `Metadata` record, or whose `Metadata` record lists no data dictionary table, is not a PFB in the format described above, and `read_pfb()` raises instead of reporting an empty table.

In [ ]:
METADATA_TABLE = "Metadata"
PATIENT_TABLE_PREFIX = "pic_sure_patients_"
DICTIONARY_TABLE_PREFIX = "pic_sure_data_dictionary_"


@dataclass
class Pfb:
    """The parts of an HPDS PFB file that the checks read.

    Attributes:
        tables: The table names the Metadata record lists.
        dictionary: The data dictionary rows.
        patient_count: The number of patient rows.
    """

    tables: list
    dictionary: list
    patient_count: int


@dataclass
class CheckResult:
    """The outcome of one check.

    Attributes:
        status: "pass", "not applicable" or "fail".
        message: One or more lines saying what was found.
    """

    status: str
    message: str


def read_pfb(path):
    """Read an HPDS PFB file into the parts the checks use.

    Args:
        path: Path of the Avro file.

    Returns:
        A Pfb holding the table names, the data dictionary rows and the
        number of patient rows.

    Raises:
        ValueError: If the file has no Metadata record, or the Metadata
            record lists no data dictionary table.
    """
    tables = None
    dictionary = []
    patient_count = 0
    with open(path, "rb") as avro_file:
        for entity in reader(avro_file):
            table = entity["name"]
            if table == METADATA_TABLE:
                tables = [node["name"] for node in entity["object"]["nodes"]]
            elif table.startswith(DICTIONARY_TABLE_PREFIX):
                dictionary.append(entity["object"])
            elif table.startswith(PATIENT_TABLE_PREFIX):
                patient_count += 1
    if tables is None:
        raise ValueError(f"{path} has no {METADATA_TABLE} record.")
    if not any(name.startswith(DICTIONARY_TABLE_PREFIX) for name in tables):
        raise ValueError(
            f"{path} has no table named {DICTIONARY_TABLE_PREFIX}<query id>. "
            f"Its tables are: {', '.join(tables)}"
        )
    return Pfb(tables, dictionary, patient_count)

In [ ]:
pfb = read_pfb(PFB_PATH)
results = {}

for table in pfb.tables:
    print(f"Table:                {table}")
print(f"Data dictionary rows: {len(pfb.dictionary)}")
print(f"Patient rows:         {pfb.patient_count}")

## Check 1: dictionary columns are filled (ALS-13635)

Every data dictionary row must have a non-empty `display` and `dataset`. A value that is missing or only whitespace counts as empty.

`description` is reported, not asserted. HPDS copies whatever the dictionary holds, and a concept may have no description.

When the notebook ran the export itself, the check also looks for a dictionary row whose `concept_path` equals each path in `CONCEPT_PATHS`.

In [ ]:
def is_blank(value):
    """Say whether a dictionary cell is missing or only whitespace."""
    return value is None or not str(value).strip()


def check_dictionary_columns(pfb, requested_paths=()):
    """Check that the data dictionary rows carry display and dataset.

    Args:
        pfb: The file to check.
        requested_paths: The concept paths the export asked for, when
            known. Each must have a data dictionary row.

    Returns:
        A CheckResult. It fails when a row has an empty display or dataset
        or a requested concept has no row, and is not applicable when the
        table is empty.
    """
    if not pfb.dictionary:
        return CheckResult(
            "not applicable",
            "The data dictionary table is empty, so no concept was exported.",
        )
    rows_by_path = {row["concept_path"]: row for row in pfb.dictionary}
    problems = []
    for column in ("display", "dataset"):
        blank = [
            path for path, row in rows_by_path.items() if is_blank(row.get(column))
        ]
        if blank:
            problems.append(
                f"{len(blank)} rows have an empty {column}: {', '.join(blank[:5])}"
            )
    missing = [path for path in requested_paths if path not in rows_by_path]
    if missing:
        problems.append(
            f"{len(missing)} requested concepts have no row: {', '.join(missing[:5])}"
        )
    without_description = sum(
        is_blank(row.get("description")) for row in pfb.dictionary
    )
    description_note = (
        f"Rows with no description: {without_description} of {len(pfb.dictionary)}."
    )
    if problems:
        return CheckResult("fail", "\n".join(problems + [description_note]))
    return CheckResult(
        "pass",
        f"All {len(pfb.dictionary)} rows have a display and a dataset. "
        + description_note,
    )

In [ ]:
dictionary_table = pd.DataFrame(
    pfb.dictionary,
    columns=["concept_path", "display", "dataset", "description", "drs_uri"],
)
display(dictionary_table)

results["ALS-13635"] = check_dictionary_columns(
    pfb,
    CONCEPT_PATHS if session is not None else (),
)
print(results["ALS-13635"].message)
assert results["ALS-13635"].status != "fail", results["ALS-13635"].message

## Check 2: DRS URIs (ALS-13611)

`drs_uri` on a data dictionary row is an array of strings. HPDS fills it from the `drs_uri` metadata of the concept in the dictionary service, and leaves it empty when the concept has none.

An export where no row has a DRS URI is reported as "not applicable", not as a failure. It means the dictionary in this environment has no `drs_uri` metadata for the exported concepts. Whether the dictionary data should carry it is tracked as ALS-13694.

In [ ]:
def check_drs_uris(pfb):
    """Check the drs_uri arrays of the data dictionary rows.

    Args:
        pfb: The file to check.

    Returns:
        A tuple of the CheckResult and a DataFrame of the concepts whose
        drs_uri is not empty, with the number of URIs each has. The result
        fails when a row's drs_uri is not an array, and is not applicable
        when no row has a URI.
    """
    populated = []
    malformed = []
    for row in pfb.dictionary:
        uris = row.get("drs_uri")
        if not isinstance(uris, list):
            malformed.append(row["concept_path"])
            continue
        uris = [uri for uri in uris if not is_blank(uri)]
        if uris:
            populated.append(
                {
                    "concept_path": row["concept_path"],
                    "drs_uri_count": len(uris),
                    "drs_uri": uris,
                }
            )
    table = pd.DataFrame(
        populated, columns=["concept_path", "drs_uri_count", "drs_uri"]
    )
    if malformed:
        return (
            CheckResult(
                "fail",
                f"{len(malformed)} rows have a drs_uri that is not an array: "
                f"{', '.join(malformed[:5])}",
            ),
            table,
        )
    if not populated:
        return (
            CheckResult(
                "not applicable",
                f"None of the {len(pfb.dictionary)} data dictionary rows has a "
                "DRS URI. The dictionary in this environment has no drs_uri "
                "metadata for these concepts. That is a data question tracked "
                "as ALS-13694, not a failure of this notebook.",
            ),
            table,
        )
    return (
        CheckResult(
            "pass",
            f"Data dictionary rows with a DRS URI: {len(populated)} of "
            f"{len(pfb.dictionary)}. URIs in all: "
            f"{int(table['drs_uri_count'].sum())}.",
        ),
        table,
    )

In [ ]:
results["ALS-13611"], drs_table = check_drs_uris(pfb)

if not drs_table.empty:
    display(drs_table)
print(results["ALS-13611"].message)
assert results["ALS-13611"].status != "fail", results["ALS-13611"].message

## Summary

One line per check. "not applicable" means the export held nothing that check could test, which the messages above explain. A check missing from `results` did not run.

In [ ]:
CHECKS = {
    "ALS-13635": "dictionary columns are filled",
    "ALS-13611": "DRS URIs",
}

for ticket, title in CHECKS.items():
    result = results.get(ticket)
    status = result.status if result else "not run"
    print(f"{ticket}  {title:<32} {status}")